# Orthographically-informed ASR evaluation

Metrics for Indic ASR evaluation that preserve legitimate orthographic alternatives.  The OIWER design follows AI4Bharat's [reference implementation](https://github.com/AI4Bharat/OIWER-Orthographically-Informed-Benchmarking-for-ASR), while the optional LLM judge interface follows Sarvam's [llm_wer](https://github.com/sarvamai/llm_wer) methodology.

In [ ]:
# | default_exp evaluation

# | export
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Optional, Sequence
import unicodedata


OIWER_REFERENCE_IMPLEMENTATION = (
    'https://github.com/AI4Bharat/OIWER-Orthographically-Informed-Benchmarking-for-ASR'
)
LLM_WER_PROMPT_SOURCE = 'https://github.com/sarvamai/llm_wer/blob/main/prompt_template.txt'


def normalize_for_evaluation(text: str) -> str:
    """Apply the NFC and whitespace normalization recommended for ASR evaluation."""
    return ' '.join(unicodedata.normalize('NFC', text).split())


def _tokens(text: str) -> list[str]:
    return normalize_for_evaluation(text).split()


@dataclass(frozen=True)
class Alignment:
    """An edit-distance alignment and its operation counts."""

    reference: tuple[str, ...]
    hypothesis: tuple[str, ...]
    operations: tuple[str, ...]

    @property
    def insertions(self) -> int:
        return self.operations.count('i')

    @property
    def deletions(self) -> int:
        return self.operations.count('d')

    @property
    def substitutions(self) -> int:
        return self.operations.count('s')

    @property
    def errors(self) -> int:
        return self.insertions + self.deletions + self.substitutions


def _levenshtein_alignment(reference: Sequence[str], hypothesis: Sequence[str]) -> Alignment:
    costs = [[0] * (len(hypothesis) + 1) for _ in range(len(reference) + 1)]
    back = [[None] * (len(hypothesis) + 1) for _ in range(len(reference) + 1)]
    for i in range(1, len(reference) + 1):
        costs[i][0], back[i][0] = i, 'd'
    for j in range(1, len(hypothesis) + 1):
        costs[0][j], back[0][j] = j, 'i'
    for i, ref_word in enumerate(reference, 1):
        for j, hyp_word in enumerate(hypothesis, 1):
            candidates = [
                (costs[i - 1][j - 1] + (ref_word != hyp_word), 'c' if ref_word == hyp_word else 's'),
                (costs[i - 1][j] + 1, 'd'),
                (costs[i][j - 1] + 1, 'i'),
            ]
            costs[i][j], back[i][j] = min(candidates, key=lambda item: item[0])
    aligned_ref, aligned_hyp, operations = [], [], []
    i, j = len(reference), len(hypothesis)
    while i or j:
        operation = back[i][j]
        if operation in ('c', 's'):
            aligned_ref.append(reference[i - 1]); aligned_hyp.append(hypothesis[j - 1]); i -= 1; j -= 1
        elif operation == 'd':
            aligned_ref.append(reference[i - 1]); aligned_hyp.append(''); i -= 1
        else:
            aligned_ref.append(''); aligned_hyp.append(hypothesis[j - 1]); j -= 1
        operations.append(operation)
    return Alignment(tuple(reversed(aligned_ref)), tuple(reversed(aligned_hyp)), tuple(reversed(operations)))


In [ ]:
# | export
def wer(reference: str, hypothesis: str) -> float:
    """Return conventional WER as a fraction, after Unicode NFC normalization."""
    reference_tokens = _tokens(reference)
    return _levenshtein_alignment(reference_tokens, _tokens(hypothesis)).errors / len(reference_tokens) if reference_tokens else 0.0


def oiwer_alignment(
    hypothesis: str,
    reference_variations: Sequence[Sequence[str]],
) -> Alignment:
    """Align a hypothesis with ordered OI reference segments.

    Each inner sequence contains valid alternatives for one reference span.  An
    alternative may contain several words, which supports compound splitting,
    merging, acronyms, and inverse-text-normalization variants.
    """
    if any(not variations for variations in reference_variations):
        raise ValueError('every reference segment must contain at least one variation')
    hypothesis_tokens = _tokens(hypothesis)
    states: list[list[Optional[tuple[int, tuple[tuple[str, ...], ...]]]]] = [
        [None] * (len(hypothesis_tokens) + 1) for _ in range(len(reference_variations) + 1)
    ]
    states[0][0] = (0, ())
    for segment_index, variations in enumerate(reference_variations, 1):
        normalized_variations = [tuple(_tokens(variation)) for variation in variations]
        for end in range(len(hypothesis_tokens) + 1):
            candidates = []
            for start in range(end + 1):
                previous = states[segment_index - 1][start]
                if previous is None:
                    continue
                for variation in normalized_variations:
                    local = _levenshtein_alignment(variation, hypothesis_tokens[start:end])
                    candidates.append((previous[0] + local.errors, previous[1] + ((variation, hypothesis_tokens[start:end]),)))
            if candidates:
                states[segment_index][end] = min(candidates, key=lambda item: item[0])
    final_candidates = []
    for end, state in enumerate(states[-1]):
        if state is not None:
            suffix = tuple(hypothesis_tokens[end:])
            final_candidates.append((state[0] + len(suffix), state[1] + (((), suffix),)))
    if not final_candidates:
        return Alignment((), tuple(hypothesis_tokens), tuple('i' for _ in hypothesis_tokens))
    _, spans = min(final_candidates, key=lambda item: item[0])
    aligned_ref, aligned_hyp, operations = [], [], []
    for reference_tokens, hypothesis_span in spans:
        local = _levenshtein_alignment(reference_tokens, hypothesis_span)
        aligned_ref.extend(local.reference); aligned_hyp.extend(local.hypothesis); operations.extend(local.operations)
    return Alignment(tuple(aligned_ref), tuple(aligned_hyp), tuple(operations))


def oiwer(
    hypothesis: str,
    reference_variations: Sequence[Sequence[str]],
    reference: Optional[str] = None,
) -> float:
    """Return Orthographically-Informed WER as a fraction.

    ``reference`` fixes the denominator to the original transcript, as in an
    OI benchmark.  When omitted, the first alternative in every segment is
    treated as that original transcript.
    """
    if reference is None:
        reference = ' '.join(variations[0] for variations in reference_variations)
    reference_length = len(_tokens(reference))
    return oiwer_alignment(hypothesis, reference_variations).errors / reference_length if reference_length else 0.0


def llm_wer(
    reference: str, hypothesis: str, judge: Callable[[str, str], bool]
) -> float:
    """Return Sarvam-style LLM-WER using an application-supplied conservative judge.

    The judge receives each non-exact aligned span and must return ``True``
    only when it is certainly semantically and phonetically equivalent.
    The final numerator is capped at the reference length, preventing repeated
    hallucinations from dominating aggregate results.
    """
    reference_tokens = _tokens(reference)
    if not reference_tokens:
        return 0.0
    alignment = _levenshtein_alignment(reference_tokens, _tokens(hypothesis))
    errors = sum(
        operation != 'c' and not judge(ref_word, hyp_word)
        for ref_word, hyp_word, operation in zip(alignment.reference, alignment.hypothesis, alignment.operations)
    )
    return min(errors, len(reference_tokens)) / len(reference_tokens)


In [ ]:
assert wer('वह डॉक्टर के पास गया', 'वह doctor के पास गया') == 0.2
variations = [['वह'], ['डॉक्टर', 'doctor'], ['के'], ['पास'], ['गया']]
assert oiwer('वह doctor के पास गया', variations) == 0.0
assert oiwer('मुझे 56849 चाहिए', [['मुझे'], ['five six eight four nine', '56849'], ['चाहिए']], 'मुझे five six eight four nine चाहिए') == 0.0
assert oiwer('one two three four', [['one two', '12'], ['three four', '34']]) == 0.0
assert llm_wer('नहीं', 'नहीं नहीं नहीं नहीं', lambda _ref, _hyp: False) == 1.0
assert llm_wer('वह डॉक्टर के पास गया', 'वह doctor के पास गया', lambda ref, hyp: {ref, hyp} == {'डॉक्टर', 'doctor'}) == 0.0
